In [1]:
import pandas as pd
import numpy as np
import rasterio
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.stats import mode
from scipy.spatial import cKDTree
import warnings
import os
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)
pd.set_option('display.precision', 3)
pd.set_option('mode.chained_assignment', None)

!rm -rf /kaggle/working/*

In [3]:
train=pd.read_csv("/kaggle/input/competitions/geohab-mlwg-competition-2026/train.csv")

test=pd.read_csv("/kaggle/input/competitions/geohab-mlwg-competition-2026/test.csv")

sub=pd.read_csv("/kaggle/input/competitions/geohab-mlwg-competition-2026/sample_submission.csv")

print(f"Train Data Shape: {train.shape}")
print(f"Check out Null Values: {train.isnull().sum()}")
print(f"Train Data info: {train.info()}")

print("#"*100)
print("#"*100)
print("#"*100)
print("#"*100)
print(f"Test Data Shape: {test.shape}")
print(f"Check out Null Values: {test.isnull().sum()}")
print(f"Test Data info: {test.info()}")

Train Data Shape: (6256, 3)
Check out Null Values: class    0
x        0
y        0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6256 entries, 0 to 6255
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   class   6256 non-null   object 
 1   x       6256 non-null   float64
 2   y       6256 non-null   float64
dtypes: float64(2), object(1)
memory usage: 146.8+ KB
Train Data info: None
####################################################################################################
####################################################################################################
####################################################################################################
####################################################################################################
Test Data Shape: (98, 3)
Check out Null Values: ID    0
x     0
y     0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeInde

In [4]:
train.head()

,class,x,y
0,NVB,453594.477,5.679e+06
1,FMAT,453561.906,5.679e+06
2,ALG,453744.452,5.679e+06
3,ALG,453863.445,5.679e+06
4,ALG,453964.612,5.679e+06


In [5]:
test.head()

,ID,x,y
0,1,453702.167,5.679e+06
1,2,454126.253,5.679e+06
2,3,453957.881,5.679e+06
3,4,453798.917,5.679e+06
4,5,453520.954,5.679e+06


In [6]:
bathy_path = None
back_path  = None

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        full = os.path.join(root, file)
        if file == 'bathymetry.tif':
            bathy_path = full
        if file == 'backscatter.tif':
            back_path = full

print("Bathymetry:", bathy_path)
print("Backscatter:", back_path)

Bathymetry: /kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/bathymetry.tif
Backscatter: /kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/backscatter.tif


In [7]:
def extract_raster_values(df, raster_path, col_name):
    coords = list(zip(df['x'], df['y']))
    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        values = [val[0] for val in src.sample(coords)]
    df[col_name] = values
    if nodata is not None:
        df[col_name] = df[col_name].replace(nodata, np.nan)
    return df

for df in [train, test]:
    extract_raster_values(df, bathy_path, 'bathymetry')
    extract_raster_values(df, back_path,  'backscatter')

print(train[['bathymetry', 'backscatter']].describe())
print("Bathy nulls:", train['bathymetry'].isna().sum())
print("Back nulls:",  train['backscatter'].isna().sum())

       bathymetry  backscatter
count    6256.000     6256.000
mean       -8.568      -23.905
std         2.939        4.177
min       -20.248      -39.140
25%       -10.757      -26.859
50%        -8.689      -24.020
75%        -6.291      -21.500
max        -1.120      -12.049
Bathy nulls: 0
Back nulls: 0


In [8]:
def add_features(df, train_ref, k_neighbors=20):
    cx = train_ref['x'].mean()
    cy = train_ref['y'].mean()

    df['x_norm'] = df['x'] - cx
    df['y_norm'] = df['y'] - cy
    df['r']      = np.sqrt(df['x_norm']**2 + df['y_norm']**2)
    df['theta']  = np.arctan2(df['y_norm'], df['x_norm'])
    df['xy']     = df['x'] * df['y']
    df['x2']     = df['x'] ** 2
    df['y2']     = df['y'] ** 2
    df['x3']     = df['x'] ** 3
    df['y3']     = df['y'] ** 3
    df['x2y']    = df['x'] ** 2 * df['y']
    df['xy2']    = df['x'] * df['y'] ** 2

    df['bathy_back_ratio'] = df['bathymetry'] / (df['backscatter'].abs() + 1e-6)
    df['bathy_back_prod']  = df['bathymetry'] * df['backscatter']
    df['bathy_back_diff']  = df['bathymetry'] - df['backscatter']
    df['bathy2']           = df['bathymetry'] ** 2
    df['back2']            = df['backscatter'] ** 2
    df['bathy_log']        = np.log1p(df['bathymetry'].abs())
    df['back_log']         = np.log1p(df['backscatter'].abs())

    tree = cKDTree(train_ref[['x', 'y']].values)
    dists, idxs = tree.query(df[['x', 'y']].values, k=k_neighbors + 1)
    dists = dists[:, 1:]
    idxs  = idxs[:, 1:]

    bathy_nb = train_ref['bathymetry'].values[idxs]
    back_nb  = train_ref['backscatter'].values[idxs]

    df['bathy_knn_mean']  = np.nanmean(bathy_nb, axis=1)
    df['bathy_knn_std']   = np.nanstd(bathy_nb,  axis=1)
    df['bathy_knn_min']   = np.nanmin(bathy_nb,  axis=1)
    df['bathy_knn_max']   = np.nanmax(bathy_nb,  axis=1)
    df['bathy_knn_range'] = df['bathy_knn_max'] - df['bathy_knn_min']
    df['bathy_knn_skew']  = (bathy_nb - np.nanmean(bathy_nb, axis=1, keepdims=True)).mean(axis=1)

    df['back_knn_mean']   = np.nanmean(back_nb, axis=1)
    df['back_knn_std']    = np.nanstd(back_nb,  axis=1)
    df['back_knn_min']    = np.nanmin(back_nb,  axis=1)
    df['back_knn_max']    = np.nanmax(back_nb,  axis=1)
    df['back_knn_range']  = df['back_knn_max'] - df['back_knn_min']

    df['knn_mean_dist']   = np.mean(dists, axis=1)
    df['knn_min_dist']    = np.min(dists,  axis=1)
    df['slope_proxy']     = df['bathy_knn_range'] / (df['knn_mean_dist'] + 1e-6)
    df['rugosity']        = df['bathy_knn_std'] / (df['bathy_knn_mean'].abs() + 1e-6)

    return df

train = add_features(train, train_ref=train, k_neighbors=20)
test  = add_features(test,  train_ref=train, k_neighbors=20)

print("Train shape after features:", train.shape)

Train shape after features: (6256, 38)


In [9]:
le = LabelEncoder()
train['target'] = le.fit_transform(train['class'])
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

exclude  = {'class', 'target', 'ID'}
features = [c for c in train.columns if c not in exclude]

X        = train[features].fillna(-9999)
y        = train['target']
X_test   = test[features].fillna(-9999)
n_classes = len(le.classes_)

print("Features:", len(features))
print("X:", X.shape, "X_test:", X_test.shape)

Class mapping: {'ALG': np.int64(0), 'FMAT': np.int64(1), 'NVB': np.int64(2), 'SGAM': np.int64(3), 'SGZ': np.int64(4)}
Features: 37
X: (6256, 37) X_test: (98, 37)


In [10]:
N_SPLITS = 10
SEED     = 42

xgb_params = dict(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.2,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
    device='cuda',
    random_state=SEED,
    objective='multi:softprob',
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    verbosity=1
)

lgb_params = dict(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    device='gpu',
    gpu_platform_id=0,
    gpu_device_id=0,
    random_state=SEED,
    verbose=-1
)

cat_params = dict(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    random_seed=SEED,
    task_type='GPU',
    l2_leaf_reg=5,
    bagging_temperature=0.7,
    verbose=False,
    early_stopping_rounds=50
)

rf_params = dict(
    n_estimators=500,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=SEED
)

et_params = dict(
    n_estimators=500,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=SEED
)

knn_params = dict(
    n_neighbors=10,
    weights='distance',
    n_jobs=-1
)

mlp_params = dict(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    max_iter=300,
    random_state=SEED,
    early_stopping=True,
    learning_rate_init=0.001,
    batch_size=64
)

In [11]:
model_names = ['xgb', 'lgb', 'cat', 'rf', 'et', 'knn', 'mlp']
oof   = {m: np.zeros((len(X), n_classes)) for m in model_names}
tpred = {m: np.zeros((len(X_test), n_classes)) for m in model_names}

skf    = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
scaler = StandardScaler()
Xs     = scaler.fit_transform(X)
Xts    = scaler.transform(X_test)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr,  X_val  = X.iloc[tr_idx],  X.iloc[val_idx]
    Xs_tr, Xs_val = Xs[tr_idx],      Xs[val_idx]
    y_tr,  y_val  = y.iloc[tr_idx],  y.iloc[val_idx]

    m = XGBClassifier(**xgb_params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof['xgb'][val_idx] = m.predict_proba(X_val)
    tpred['xgb']       += m.predict_proba(X_test) / N_SPLITS

    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)])
    oof['lgb'][val_idx] = m.predict_proba(X_val)
    tpred['lgb']       += m.predict_proba(X_test) / N_SPLITS

    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
    oof['cat'][val_idx] = m.predict_proba(X_val)
    tpred['cat']       += m.predict_proba(X_test) / N_SPLITS

    m = RandomForestClassifier(**rf_params)
    m.fit(X_tr, y_tr)
    oof['rf'][val_idx] = m.predict_proba(X_val)
    tpred['rf']       += m.predict_proba(X_test) / N_SPLITS

    m = ExtraTreesClassifier(**et_params)
    m.fit(X_tr, y_tr)
    oof['et'][val_idx] = m.predict_proba(X_val)
    tpred['et']       += m.predict_proba(X_test) / N_SPLITS

    m = KNeighborsClassifier(**knn_params)
    m.fit(Xs_tr, y_tr)
    oof['knn'][val_idx] = m.predict_proba(Xs_val)
    tpred['knn']       += m.predict_proba(Xts) / N_SPLITS

    m = MLPClassifier(**mlp_params)
    m.fit(Xs_tr, y_tr)
    oof['mlp'][val_idx] = m.predict_proba(Xs_val)
    tpred['mlp']       += m.predict_proba(Xts) / N_SPLITS

    fold_ens = sum(oof[m][val_idx] for m in model_names) / len(model_names)
    f1 = f1_score(y_val, np.argmax(fold_ens, axis=1), average='weighted')
    print(f"Fold {fold+1:2d} | F1: {f1:.4f}")

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Fold  1 | F1: 0.9888
Fold  2 | F1: 0.9840
Fold  3 | F1: 0.9905
Fold  4 | F1: 0.9920
Fold  5 | F1: 0.9873
Fold  6 | F1: 0.9904
Fold  7 | F1: 0.9904
Fold  8 | F1: 0.9823
Fold  9 | F1: 0.9872
Fold 10 | F1: 0.9936


In [12]:
model_f1 = {}
for m in model_names:
    f1 = f1_score(y, np.argmax(oof[m], axis=1), average='weighted')
    model_f1[m] = f1
    print(f"{m.upper():5s}: {f1:.4f}")

total_f1 = sum(model_f1.values())
weights = {m: model_f1[m] / total_f1 for m in model_names}

print("\nWeights:")
for m, w in weights.items():
    print(f"  {m.upper():5s}: {w:.4f}")

oof_ensemble = np.zeros_like(oof[model_names[0]])
for m in model_names:
    oof_ensemble += oof[m] * weights[m]

oof_f1 = f1_score(y, np.argmax(oof_ensemble, axis=1), average='weighted')
print(f"\nWeighted Ensemble OOF F1: {oof_f1:.4f}")

XGB  : 0.9893
LGB  : 0.9888
CAT  : 0.9880
RF   : 0.9886
ET   : 0.9883
KNN  : 0.9512
MLP  : 0.9727

Weights:
  XGB  : 0.1441
  LGB  : 0.1440
  CAT  : 0.1439
  RF   : 0.1440
  ET   : 0.1439
  KNN  : 0.1385
  MLP  : 0.1416

Weighted Ensemble OOF F1: 0.9886


In [13]:
test_ensemble = np.zeros_like(tpred[model_names[0]])
for m in model_names:
    test_ensemble += tpred[m] * weights[m]

final_preds = np.argmax(test_ensemble, axis=1)
sub['class'] = le.inverse_transform(final_preds)
sub.to_csv("submission.csv", index=False)

print(sub['class'].value_counts())
sub.head()

class
NVB     37
ALG     27
FMAT    18
SGZ      9
SGAM     7
Name: count, dtype: int64


,ID,class
0,1,NVB
1,2,ALG
2,3,NVB
3,4,NVB
4,5,SGZ
